In [167]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException, ElementNotInteractableException, ElementClickInterceptedException   
from datetime import datetime
import traceback

import pandas as pd
import re
import numpy as np
from time import sleep
import urllib.parse

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

In [168]:
# Set up functions
def print_update(func):
    def wrapper(*args, **kwargs):
        print(f'Running {func.__name__}')
        return func(*args, **kwargs)
    return wrapper


def reject_cookies(driver, temp=0, retries=2):
    try:
        WebDriverWait(driver, retries).until(EC.element_to_be_clickable((By.ID, 'onetrust-reject-all-handler')))
        driver.find_element(By.ID, 'onetrust-reject-all-handler').click()
    except (NoSuchElementException, TimeoutException, ElementNotInteractableException, ElementClickInterceptedException):
        if temp >= retries:
            print(f'{retries} attempts failed to Reject cookies')
        else:
            reject_cookies(driver, temp + 1)


def skip_tutorial(driver, temp=0, retries=2):
    try:
        WebDriverWait(driver, retries).until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'walla-button.TooltipWrapper__skip')))
        driver.find_element(By.CSS_SELECTOR, 'walla-button.TooltipWrapper__skip').click()
        driver.find_element(By.CSS_SELECTOR, 'walla-button.TooltipWrapper__skip').click()
        driver.find_element(By.CSS_SELECTOR, 'walla-button.TooltipWrapper__skip').click()
    except (NoSuchElementException, TimeoutException, ElementNotInteractableException, ElementClickInterceptedException):
        if temp >= retries:
            print(f'{temp} attempts failed to Skip the tutorial')
        else:
            skip_tutorial(driver, temp + 1)

@print_update
def setup_driver():
    options = webdriver.ChromeOptions()
    prefs = {"profile.managed_default_content_settings.images": 2}
    options.add_experimental_option("prefs", prefs)
    # options.add_argument("--headless")
    options.add_argument("--window-size=1920,1080")
    options.add_argument("--log-level=4")
    options.add_argument("--disable-extensions")
    driver = webdriver.Chrome(options=options)
    driver.maximize_window()

    driver.get('https://es.wallapop.com/app/search?filters_source=category_navigation&latitude=40.41956&longitude=-3.69196')

    reject_cookies(driver)
    skip_tutorial(driver)

    return driver

@print_update
def get_explorer_urls(product_id=None, category_id=24200):
    # gets the corresponding urls to wallapop listing explorer
    base = 'https://es.wallapop.com/app/search?'
    urls = []

    if product_id:
        pass ## Do later
        # url = url if (product_id == None) else url + f'keywords={urllib.parse.quote(product_id)}&'
    else:
        '''
        Make a df for every category 
        Get the histogram curve for each category type by querying the listings db
        '''


        if category_id == 24200:
            object_type_ids = [9447, 9487, 10175, 24102]
            object_type_ids = [9447]
            for object_id in object_type_ids:
                url = base + f'category_ids={category_id}&filters_source=quick_filters&latitude=40.41956&longitude=-3.69196&object_type_ids={object_id}&'
                urls += split_explorer_url(url, product_id=product_id, category_id=category_id, object_id=object_id)
        else:
            print('category not supported')

    return urls


def split_explorer_url(url, product_id=None, category_id=None, object_id=None):
    # breaks up url into smaller subsets to divide up scrape load
    urls = []

    if product_id:
        # IGNORE write later after figuring out product_id classification module
        pass
    else:
        start, increment, loops = 0, 4, 12
        current_range = increment

        for i in range(loops):
            min = start
            max = start + current_range

            if i == 0:
                pricequery = f'max_sale_price={max}'
            elif i + 1 == len(range(loops)):
                pricequery = f'min_sale_price={min+.01}'
            else:
                pricequery = f'min_sale_price={min+.01}&max_sale_price={max}'

            start += current_range
            current_range += increment + (i-1)**2 # Ramps up range ~according to previous scrapes

            urls.append({
                'explorer_url': url + pricequery,
                'product_id': product_id,
                'category_id': category_id,
                'object_id': object_id,
                'date_scraped': np.nan,
                'scrape_empty': np.nan,
                'scrape_incomplete': np.nan,
                'redundant_url': np.nan,
            })
    return urls


def load_more(driver, temp=0, retries=2):
    # Clicks button that initiates infinite scroll dynamic listings
    try:
        WebDriverWait(driver, retries).until(EC.element_to_be_clickable((By.ID, 'btn-load-more')))
        driver.find_element(By.ID, 'btn-load-more').click()
    except (NoSuchElementException, TimeoutException, ElementNotInteractableException, ElementClickInterceptedException):
        if temp >= retries:
            print(f'{temp} attempts failed to Click the load button')
        else:
            load_more(driver, temp + 1)


def wait_listings(driver, temp=0, retries=3):
    try:
        WebDriverWait(driver, retries).until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'a.ItemCardList__item')))
        return True
    except (NoSuchElementException, TimeoutException, ElementNotInteractableException, ElementClickInterceptedException):
        if temp >= retries:
            print(f'{temp} attempts failed to Find listings')
        else:
            wait_listings(driver, temp + 1)

In [172]:
df = pd.read_csv('data/categories/24200/10414.csv')
df[df['date_last_scraped'] != df['date_first_scraped']]

,title,https://es.wallapop.com/item/,object_id,listing_id,num_images,date_last_scraped,date_first_scraped,price_cents,featured,shipping,reserved,walla_pro


In [170]:
# Explorer function
@print_update
def scrape_explorer(driver, object_id=None):
    # Scrapes listings displayed in explorer
    scrapings = []
    cards = driver.find_elements(By.CSS_SELECTOR, 'a.ItemCardList__item')
    for card in cards:
        title = card.get_attribute('title')
        href = card.get_attribute('href')
        href = href.split('https://es.wallapop.com/item/')[-1]
        listing_id = href.split('-')[-1]
        num_images = len(card.find_elements(By.TAG_NAME, 'li'))
        if not num_images:
            num_images = 1
        date_scraped = datetime.now().strftime('%d%m%y%H%M%S')

        price = card.find_element(By.CSS_SELECTOR, 'span.ItemCard__price').text.strip()
        price = float(re.sub(r'[^\d,]', '', price).replace(',', '.'))
        price = int(price*100)
        
        try:
            if card.find_element(By.CSS_SELECTOR, 'p.ItemCard__highlight-text.pb-2'):
                featured = 1
            else:
                featured = 0
        except NoSuchElementException:
            featured = 0

        shadow_hosts = card.find_elements(By.CSS_SELECTOR, 'wallapop-badge')
        shipping, reserved, walla_pro = 0, 0, 0
        for shadow_host in shadow_hosts:
            shadow_root = driver.execute_script('return arguments[0].shadowRoot', shadow_host)
            span = shadow_root.find_element(By.CSS_SELECTOR, 'span').text.strip() 
            if span == 'Sólo venta en persona':
                shipping = 0
            elif span == 'Envío disponible':
                shipping = 1
            elif span == 'Envío gratis':
                shipping = 2
            elif span == 'Reservado':
                reserved = 1
            elif span == 'Reacondicionado':
                walla_pro = 1
            else:
                print('\nUNEXPECTED SHADOW_ROOT DATA\n')

        scrapings.append({
            'title': title,
            'https://es.wallapop.com/item/': href,
            'object_id': object_id,
            'listing_id': listing_id,
            'num_images': num_images,
            'date_last_scraped': date_scraped,
            'date_first_scraped': date_scraped,
            'price_cents': price,
            'featured': featured,
            'shipping': shipping,
            'reserved': reserved,
            'walla_pro': walla_pro,
        })
        
    return pd.DataFrame(scrapings)

@print_update
def scroll_explorer(driver, max_scrolls=50, category_id=None, object_id=None):
    global scraped_df
    previous_height = driver.execute_script("return document.body.scrollHeight")
    sleep_time = 0.1
    count = 0
    timer = datetime.now()
    complete = False

    while True:
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        count += 1
        if count == max_scrolls:
            print('    Scroll: Count reached')
            scraped_df, complete = check_scraping(driver, False, timer, category_id=category_id, object_id=object_id)

        try:
            WebDriverWait(driver, sleep_time).until(lambda driver: driver.execute_script("return document.body.scrollHeight") != previous_height)
            sleep_time /= 2
            previous_height = driver.execute_script("return document.body.scrollHeight")
        except TimeoutException:
            sleep_time += sleep_time
            if sleep_time > 12:
                print('    Scroll: Timed out')
                scraped_df, complete = check_scraping(driver, True, timer, category_id=category_id, object_id=object_id)

        if complete:
            return scraped_df, True

@print_update
def check_scraping(driver, timed_out, timer=None, category_id=None, object_id=None):
    if timed_out:
        pass
    else:
        sleep(1)
        
    # Get scrapings and check if they already exist in global
    scrapings_df = scrape_explorer(driver, object_id=object_id)

    if scrapings_df.empty:
        return scraped_df, True
    else:
        scraped_df = update_scraping(scrapings_df, category_id=category_id, object_id=object_id)
        clear_explorer(driver)

        # # Print stats
        timespent = datetime.now() - timer
        print(f'    scrapings took: {timespent}')
        print(f'    scrapings count: {len(scrapings_df)}')
        print(f'    scrapings efficiency = {len(scrapings_df)/timespent.total_seconds()}')
        if timed_out:
            return scraped_df, True
        else:
            return scroll_explorer(driver, category_id=category_id, object_id=object_id)


# def update_scraping(scrapings_df, category_id=None, object_id=None):
#     try:
#         object_df = pd.read_csv(f'data/categories/{category_id}/{object_id}.csv')
#         if scrapings_df['listing_id'].isin(object_df['listing_id']).all():
#             print('Scroll: Seen before')
#             return scraped_df
#         else:
#             object_df = pd.concat([object_df, scrapings_df]).drop_duplicates()
#             object_df.to_csv(f'data/categories/{category_id}/{object_id}.csv', index=False)

#         if scraped_df.empty:
#             scraped_df = scrapings_df
#             return scraped_df
#         else:
#             scraped_df = pd.concat([scraped_df, scrapings_df]).drop_duplicates()
#             return scraped_df

#     except FileNotFoundError:
#         if scraped_df.empty:
#             scraped_df = scrapings_df
#             scraped_df.to_csv(f'data/categories/{category_id}/{object_id}.csv', index=False)
#             return scraped_df
#         else:
#             scraped_df = pd.concat([scraped_df, scrapings_df]).drop_duplicates()
#             scraped_df.to_csv(f'data/categories/{category_id}/{object_id}.csv', index=False)
#             return scraped_df

@print_update
def update_scraping(scrapings_df, category_id=None, object_id=None):
    try:
        object_df = pd.read_csv(f'data/categories/{category_id}/{object_id}.csv')
        ignore_columns = ['date_first_scraped', 'date_last_scraped']
        compare_columns = [col for col in scrapings_df.columns if col not in ignore_columns]

        for _, row in scrapings_df.iterrows():
            match = object_df[compare_columns].eq(row[compare_columns]).all(axis=1)
            print(match)
            if match.any():
                object_df.loc[match, 'date_last_scraped'] = datetime.now().strftime('%d%m%y%H%M%S')
            else:
                object_df = pd.concat([object_df, pd.DataFrame([row])], ignore_index=True)
        
        object_df.to_csv(f'data/categories/{category_id}/{object_id}.csv', index=False)

        if 'scraped_df' not in globals():
            
            scraped_df = scrapings_df
        else:
            scraped_df = pd.concat([scraped_df, scrapings_df]).drop_duplicates()
        
        return scraped_df

    except FileNotFoundError:
        object_df = scrapings_df
        object_df.to_csv(f'data/categories/{category_id}/{object_id}.csv', index=False)
        return scrapings_df



@print_update
def clear_explorer(driver):
    driver.execute_script("window.scrollTo(0, 0);")
    try:
        # Find elements and delete all siblings
        element = driver.find_element(By.CSS_SELECTOR, 'a.ItemCardList__item')
        parent = element.find_element(By.XPATH, '..')
        siblings = parent.find_elements(By.XPATH, '*')
        for sibling in siblings:
            driver.execute_script("arguments[0].remove();", sibling)
    except NoSuchElementException:
        print('    No listings found to clear to clear')
   

def check_newest(category_ids=[24200, 12579, 13100]):
    for category_id in category_ids:
        category_start = datetime.now()

        try:
            categories = pd.read_csv('data/categories/categories.csv')
            object_type_ids = categories[categories['category_id'] == 24200]
            object_type_ids = object_type_ids[object_type_ids['subgroup_name'] != 'FULL_GROUP']
            # object_type_ids = object_type_ids[object_type_ids['last_checked'] != category_start.strftime('%d%m%y%H%M%S')]
        except FileNotFoundError:
            return 'data/categories/categories.csv is missing'
        category_name = categories.loc[categories['category_id'] == category_id, 'category_name'].values[0]
        print(f'\n=== Checking {category_name}')

        scraped_count = 0
        for object_id in object_type_ids['object_type_id']:
            object_name = categories.loc[categories['object_type_id'] == object_id, 'subgroup_name'].values[0]
            print(f'\nSCRAPING {object_name.upper()}')

            object_start = datetime.now()
            driver = setup_driver()
            driver.get(f'https://es.wallapop.com/app/search?object_type_ids={object_id}&category_ids={category_id}&filters_source=quick_filters&latitude=40.41956&longitude=-3.69196&order_by=newest')
            if wait_listings(driver):
                try:
                    load_more(driver)
                    scraped_df = pd.DataFrame()
                    scraped_df = scroll_explorer(driver, category_id=category_id, object_id=object_id)
                    driver.quit()
                except Exception as e:
                    print(f'{category_id}-{object_id} Error')
                    traceback.print_exc()
                    driver.quit()
                    return e
            else:
                driver.quit()

            # Update categories.csv
            categories.loc[categories['object_type_id'] == object_id, 'last_checked'] = datetime.now().strftime('%d%m%y%H%M%S')
            categories.to_csv('data/categories/categories.csv', index=False)

            timespent = datetime.now() - object_start
            scraped_count += len(scraped_df)
            print(f'\n{object_name.upper()} SCRAPE COMPLETE')
            print(f'    {object_name} took: {timespent}')
            print(f'    {object_name} scraped: {len(scraped_df)}')
            print(f'    {object_name} efficency: {len(scraped_df)/timespent.total_seconds()}')


        categories.loc[(categories['category_id'] == category_id) & (categories['subgroup_name'] == 'FULL_GROUP'), 'last_checked'] = datetime.now().strftime('%d%m%y%H%M%S')
        categories.to_csv('data/categories/categories.csv', index=False)

        timespent = datetime.now() - category_start
        print(f'\n{category_name} category checked')
        print(f'    {category_name} took: {timespent}')
        print(f'    {category_name} scraped: {scraped_count}')
        print(f'    {category_name} efficency: {scraped_count/timespent.total_seconds()}')

    
    print('\n=== All categories checked ===\n')

In [171]:
categories = [24200]

while True:
    try:
        message = check_newest(categories)
        if message:
            print(message)
            break
    except Exception:
        traceback.print_exc()
    sleep(5)


=== Checking Tecnología y electrónica

SCRAPING TV
Running setup_driver
Running scroll_explorer
    Scroll: Count reached
Running check_scraping
Running scrape_explorer
Running update_scraping
0       False
1       False
2       False
3       False
4       False
        ...  
4109    False
4110    False
4111    False
4112    False
4113    False
Length: 4114, dtype: bool
0       False
1       False
2       False
3       False
4       False
        ...  
4110    False
4111    False
4112    False
4113    False
4114    False
Length: 4115, dtype: bool
0       False
1       False
2       False
3       False
4       False
        ...  
4111    False
4112    False
4113    False
4114    False
4115    False
Length: 4116, dtype: bool
0       False
1       False
2       False
3       False
4       False
        ...  
4112    False
4113    False
4114    False
4115    False
4116    False
Length: 4117, dtype: bool
0       False
1       False
2       False
3       False
4       False
        ...  
41

KeyboardInterrupt: 

In [3]:
pd.set_option('display.max_rows', None)
display(pd.read_csv('data/keys/categories.csv'))
pd.set_option('display.max_rows', 50)

,category_name,group_name,subgroup_name,category_ids,object_type_ids
0,Tecnología y electrónica,NaN,FULL_CATEGORY,24200,NaN
1,Tecnología y electrónica,Imagen,FULL_GROUP,24200,10206.0
2,Tecnología y electrónica,Imagen,TV,24200,10414.0
3,Tecnología y electrónica,Imagen,Proyectores,24200,10406.0
4,Tecnología y electrónica,Imagen,Accesorios de televisores y proyectores,24200,10405.0
5,Tecnología y electrónica,Imagen,Otros artículos de imagen,24200,24192.0
6,Tecnología y electrónica,Sonido,FULL_GROUP,24200,24149.0
7,Tecnología y electrónica,Sonido,Auriculares,24200,10395.0
8,Tecnología y electrónica,Sonido,Altavoces,24200,24150.0
9,Tecnología y electrónica,Sonido,Home cinema y barras de sonido,24200,24151.0


In [ ]:
def elaborate_item():